## Server-Side Tracking detection
- Our SST endpoint identification method focused on the server-side use of Google Tag Manager, rather than generic server-side tracking.
Our method relies on common URL parameters extracted from the data we collected; it may not generalize to other datasets or more customized uses of
SST. See §3.4 in the paper.

In [1]:
import pandas as pd
import seaborn as sns
from utils import parse_har_files_to_dataframe
from common import BASE_HAR_DIR


sns.set_theme(style="whitegrid", context="paper")
pd.set_option('display.max_colwidth', 128)

df = parse_har_files_to_dataframe(BASE_HAR_DIR, test=False)

### Identify the HTTP parameters that we use to detect SST

In [ ]:
GTM_JS_URL_PREFIX = 'https://www.googletagmanager.com/gtm.js?id=GTM-'

gtm_js_scripts_by_source = set(df[df.has_gtm_signature].url.unique())
gtm_js_scripts_by_url = set(df[df.url.str.startswith(GTM_JS_URL_PREFIX)].url.unique())
gtm_js_urls = gtm_js_scripts_by_source.union(gtm_js_scripts_by_url)

# requests that are triggered by GTM
df['triggered_by_gtm'] = df.initiator_urls.apply(
    lambda x: any([url in gtm_js_urls for url in x]))


df['has_gtm_param'] = df.param_names.apply(lambda x: 'gtm' in x)

# intersection of all GA4 parameters that are triggered by GTM
ga4_params_set = df[
    df.triggered_by_gtm &
    df.has_gtm_param &
    (df.entity != 'Google')  # exclude requests sent to Google, as we focus on SST-related first-party usage
    ]['param_names'].map(set)

from functools import reduce
ga4_intersection = reduce(set.intersection, ga4_params_set)

# label requests that contain all of the params of the intersection
df['has_intersect_ga4_params'] = df['param_names'].apply(lambda x: ga4_intersection.issubset(x))
# Display the intersection
print(len(ga4_intersection), sorted(list(ga4_intersection)))

36 ['_p', '_s', 'are', 'cid', 'dl', 'dma', 'dma_cps', 'dt', 'ecid', 'frm', 'gcd', 'gcs', 'gtm', 'npa', 'pscdl', 'sct', 'seg', 'sid', 'sr', 'sst.rnd', 'sst.tft', 'sst.ude', 'tag_exp', 'tfd', 'tid', 'uaa', 'uab', 'uafvl', 'uam', 'uamb', 'uap', 'uapv', 'uaw', 'ul', 'ur', 'v']


In [7]:
df['triggered_by_gtm'].value_counts()

triggered_by_gtm
False    230609
True      12358
Name: count, dtype: int64

## Self-hosted GTM scripts

In [8]:
df[df.has_gtm_signature & (df.entity != "Google")].url.value_counts().to_frame()

,count
url,
https://tmsst.aponeo.de/gtm.js?id=GTM-5GNVZ2M,4
https://sgtm.mycare.de/gtm.js?id=GTM-NR6HHR,4
https://measure.docmorris.de/gtm.js?id=GTM-KBGX7K8,4
https://pipeline.drogist.nl/webevents.js?id=GTM-TVJBDC,4
https://sgtm.plein.nl/load?id=GTM-5N8W39,4
https://sst.koopjesdrogisterij.nl/gtm.js?id=GTM-N45WVXP,4
https://sgtm.docpeter.it/mapwbued.js?id=GTM-KV9B4PQ,4
https://sgtm.farmasave.it/wyxyzwjg.js?id=GTM-NP6XD6D,3


## Detect SST domains hosted on Google
- We check where SST domains are hosted. This is used for verification, rather than a criterion for detection.

In [9]:
import socket
import whois
import os
import json
from functools import cache

WHOIS_CACHE_FILE = 'whois_cache.json'

@cache
def get_whois_info(host):
    ip_address = socket.gethostbyname(host)
    whois_info = whois.whois(ip_address)
    whois_org = whois_info.get('org', '') if whois_info.get('org') else ''
    whois_domains = whois_info.get('domain_name')
    if isinstance(whois_domains, list):
        whois_domains = list(set([domain.upper() for domain in whois_domains]))
        whois_domains = ' '.join(whois_domains) if whois_domains else ''
    elif whois_domains:
        whois_domains = whois_domains.upper()
    else:
        whois_domains = ''
    return whois_org.upper() + '\t' + whois_domains


def load_whois_cache(cache_file, hostnames):
    whois_dict = {}
    if os.path.exists(cache_file):
        print(f'Loading whois cache from {cache_file}')
        with open(cache_file, 'r') as f:
            return json.load(f)

    # Update the whois_dict with new entries
    for req_hostname in hostnames:
        if req_hostname not in whois_dict:
            whois_dict[req_hostname] = get_whois_info(req_hostname)
            # print(req_hostname, get_whois_info(req_hostname))
    # Save the updated cache
    with open(WHOIS_CACHE_FILE, 'w') as f:
        json.dump(whois_dict, f)

    return whois_dict



In [10]:
whois_dict = load_whois_cache(WHOIS_CACHE_FILE, df[
    df.has_intersect_ga4_params].req_hostname.unique())

df['whois_info'] = df.req_hostname.apply(lambda x: whois_dict.get(x, ''))

In [11]:
GOOGLE_SERVERS = [
    'Google Tag Manager',
    'Google Frontend'
    ]

GOOGLE_WHOIS = [
    "GOOGLE LLC-GOOGLEUSERCONTENT.COM",
    "GOOGLE LLC-1E100.NET",
    "-GOOGLEUSERCONTENT.COM"
    ]


def is_hosted_by_google(row):
    """Not used for detection"""
    whois = row['whois_info']
    if ((whois in GOOGLE_WHOIS) or
        ('1.1 google' in row['via']) or
            (row['server'].strip() in GOOGLE_SERVERS)):
        return True
    return False

###########################################################
df['google_hosted'] = df.apply(is_hosted_by_google, axis=1)
###########################################################


In [12]:
# SST = Server Side Tagging
# label requests that are triggered by (GTM or gtag) and contain all GA4 parameters
df['sst'] = df.has_intersect_ga4_params

df['url_path'] = df.url.apply(lambda x: x.split('?')[0])
df['first_party'] = ~df.third_party

dft = df[df.sst]
df.sst.value_counts()

sst
False    241409
True       1558
Name: count, dtype: int64

In [21]:
df[(df.exp_mode == 'tracking') & df.sst].drop_duplicates(COLS)[
    ['country', 'site_domain', 'req_hostname',
    'google_hosted', 'via', 'server', 'whois_info', 'cname']]

,country,site_domain,req_hostname,google_hosted,via,server,whois_info,cname
68161,DE,medpex.de,measure.medpex.de,True,1.1 google,Google Frontend,\tGOOGLEUSERCONTENT.COM,
69585,DE,aponeo.de,tmsst.aponeo.de,True,1.1 google,Google Frontend,\tGOOGLEUSERCONTENT.COM,
72546,DE,shop-apotheke.com,klpoz.shop-apotheke.com,True,1.1 google,Google Frontend,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
76257,DE,mycare.de,sgtm.mycare.de,False,,,\t,
81865,DE,docmorris.de,measure.docmorris.de,True,1.1 google,Google Frontend,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
84757,IT,redcare.it,otasf.redcare.it,True,1.1 google,Google Frontend,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
92885,IT,efarma.com,gtm.efarma.com,False,,Apache/2.4.62 (Debian),\tGROWEB.IT,ihcjdeaiapzd.eu-1.tagmanagerserverside.it
98532,IT,farmasave.it,sgtm.farmasave.it,False,,,GOOGLE LLC\tGOOGLEUSERCONTENT.COM,
100629,IT,docpeter.it,sgtm.docpeter.it,False,,,\tGOOGLEUSERCONTENT.COM,
122017,ES,farmaciasdirect.es,datos.farmaciasdirect.es,True,1.1 google,,GOOGLE LLC\t1E100.NET,ghs.googlehosted.com


## SST use by consent type

In [ ]:
COLS = ['site_domain', 'req_hostname', 'country']

dft[dft.exp_mode=='tracking'].drop_duplicates(['consent_mode', 'site_domain', 'req_hostname'])[
    ['consent_mode', 'country', 'site_domain', 'req_hostname']].\
        sort_values(['consent_mode', 'country'])

,consent_mode,country,site_domain,req_hostname
68161,Accept,DE,medpex.de,measure.medpex.de
69585,Accept,DE,aponeo.de,tmsst.aponeo.de
72546,Accept,DE,shop-apotheke.com,klpoz.shop-apotheke.com
76257,Accept,DE,mycare.de,sgtm.mycare.de
81865,Accept,DE,docmorris.de,measure.docmorris.de
122017,Accept,ES,farmaciasdirect.es,datos.farmaciasdirect.es
84757,Accept,IT,redcare.it,otasf.redcare.it
92885,Accept,IT,efarma.com,gtm.efarma.com
98532,Accept,IT,farmasave.it,sgtm.farmasave.it
100629,Accept,IT,docpeter.it,sgtm.docpeter.it


In [25]:
reject_hosts = set(df[df.sst & (df.consent_mode == 'Reject')].req_hostname.unique())
accept_hosts = set(df[df.sst & (df.consent_mode == 'Accept')].req_hostname.unique())
a_r_hosts = accept_hosts.intersection(reject_hosts)
only_accept_hosts = accept_hosts - a_r_hosts
only_reject_hosts = reject_hosts - a_r_hosts
print('Accept hosts:', len(accept_hosts), 'Reject hosts:', len(reject_hosts), 'Accept-Reject hosts:', len(a_r_hosts))
print('Only Accept hosts:', only_accept_hosts, '\nOnly Reject hosts:', only_reject_hosts)
df['gtm_consent'] = df.req_hostname.apply(lambda x: 'Only in accept' if x in only_accept_hosts else 'Only in reject' if x in only_reject_hosts else 'Both' if x in a_r_hosts else 'No GTM consent')

Accept hosts: 18 Reject hosts: 15 Accept-Reject hosts: 14
Only Accept hosts: {'sgtm.mycare.de', 'tmsst.aponeo.de', 'measure.medpex.de', 'measure.docmorris.de'} 
Only Reject hosts: {'care.soin-et-nature.com'}


## Table 4: SST endpoints by country (Loc.) and used consent mode (A: Accept, R: Reject)

In [26]:
print(df[(df.exp_mode == 'tracking') & df.sst].drop_duplicates(COLS)[
    ['country', 'url_path', 'google_hosted', 'gtm_consent']
].to_latex(index=False, header=['Country', 'SST Hostname', 'Google Hosted', 'Consent']))

\begin{tabular}{llrl}
\toprule
Country & SST Hostname & Google Hosted & Consent \\
\midrule
DE & https://measure.medpex.de/g/collect & True & Only in accept \\
DE & https://tmsst.aponeo.de/g/collect & True & Only in accept \\
DE & https://klpoz.shop-apotheke.com/g/collect & True & Both \\
DE & https://sgtm.mycare.de/g/collect & False & Only in accept \\
DE & https://measure.docmorris.de/g/collect & True & Only in accept \\
IT & https://otasf.redcare.it/g/collect & True & Both \\
IT & https://gtm.efarma.com/g/collect & False & Both \\
IT & https://sgtm.farmasave.it/ngtwyxyzwjg & False & Both \\
IT & https://sgtm.docpeter.it/ngtmapwbued & False & Both \\
ES & https://datos.farmaciasdirect.es/g/collect & True & Both \\
NL & https://pipeline.drogist.nl/g/collect & True & Both \\
NL & https://metrics.deonlinedrogist.nl/g/collect & True & Both \\
NL & https://sgtm.plein.nl/g/collect & True & Both \\
NL & https://ecom-data.trekpleister.nl/g/collect & True & Both \\
NL & https://ecom-data.krui